# 03 · Model adaptation

This notebook obtains the three pretrained foundation models and fine-tunes them on PTB-XL for the five diagnostic classes.
It writes every fine-tuned model that the paper uses:

| Models | Kinds | Seeds | Number |
|---|---|---|---|
| ECGFounder, HuBERT-ECG, ECG-FM | `baseline`, `teacher`, `student` | 0, 1, 2 | 27 |
| ECGFounder | `windows`, `agreement`, `selfdistill` (alternatives for the training remedy) | 0, 1, 2 | 9 |
| ECGFounder | `baseline` (further models for the ensemble of ten) | 3 … 9 | 7 |
| ECGFounder | `dropout10` … `dropout50` (Monte Carlo dropout) | 0, 1, 2 | 15 |

**Output files**

- `FINETUNED/{tag}.pt`: the weights of the step with the highest validation AUROC, with `tag = {model}_{kind}_s{seed}`;
- `TRAINING_LOGS/{tag}.csv`: loss and validation AUROC every 250 steps;
- `TEACHER_TARGETS/{tag}.npz`: the distillation targets calculated with the model `tag` for every training segment.

**Time** on one RTX 3080 (10 GB), 3,000 steps each:

| | baseline | teacher | teacher targets | student |
|---|---|---|---|---|
| ECGFounder | 5 min | 3 min | 2 min | 10 min |
| HuBERT-ECG | 15 min | 15 min | 6 min | 30 min |
| ECG-FM | 24 min | 13 min | 5 min | 46 min |

The 27 models of the main analysis take about 9 h, and the 31 comparison models of ECGFounder about 5 h (5 to 19 min each). A model whose file exists is skipped, so the notebook can be stopped and run again.
The fine-tuning function itself, `finetune`, is defined in notebook 00 (section *Fine-tuning*), because notebook 06 times the same function; this notebook explains the method and runs the jobs.

In [ ]:
%run ./00_setup_and_settings.ipynb

In [ ]:
QUICK_TEST = False        # True: functional check (40 steps, a few hundred recordings, seed 0 only); everything is written to subfolders "quick_test" and no model of the study is touched

SUB = "quick_test" if QUICK_TEST else ""
MODEL_DIR, LOG_DIR, TARGET_DIR = FINETUNED / SUB, TRAINING_LOGS / SUB, TEACHER_TARGETS / SUB
STEPS, EVERY, SUBSET = (40, 20, QUICK_SUBSET) if QUICK_TEST else (TRAIN_STEPS, EVAL_EVERY, None)
for _d in (MODEL_DIR, LOG_DIR, TARGET_DIR):
    _d.mkdir(parents=True, exist_ok=True)
assert DEVICE.type == "cuda", "this notebook needs a GPU"
print("GPU:", torch.cuda.get_device_name(0), "| models are written to", MODEL_DIR)

## Pretrained weights and code

The pretrained models are not part of this repository. They are obtained from their authors and placed under `MASTER/models/pretrained/`:

| Model | Source | Files | Placed in | Licence |
|---|---|---|---|---|
| ECGFounder (code) | GitHub [`PKUDigitalHealth/ECGFounder`](https://github.com/PKUDigitalHealth/ECGFounder) | the repository; only `net1d.py` (class `Net1D`) is imported | `ECGFounder/` | MIT |
| ECGFounder (weights) | Hugging Face [`PKUDigitalHealth/ECGFounder`](https://huggingface.co/PKUDigitalHealth/ECGFounder) | `12_lead_ECGFounder.pth` (0.37 GB) | `12_lead_ECGFounder.pth` | MIT |
| HuBERT-ECG, base size | Hugging Face [`Edoardo-Coppola/hubert-ecg-base`](https://huggingface.co/Edoardo-Coppola/hubert-ecg-base) | `model.safetensors` (0.37 GB), `config.json` | `hubert_ecg/` | CC BY-NC 4.0 (non-commercial use) |
| ECG-FM | Hugging Face [`wanglab/ecg-fm`](https://huggingface.co/wanglab/ecg-fm) | `mimic_iv_ecg_physionet_pretrained.pt` (1.09 GB) | `ecg_fm/` | MIT |

Notes

- **ECGFounder.** The 12-lead model, pretrained with supervision on 150 diagnostic labels. The study used commit `04edac7` of the code repository.
- **HuBERT-ECG.** The weights are loaded into the standard `HubertModel` of the `transformers` library. The model class that is published with the weights (`hubert_ecg.py`) is not downloaded and not executed.
- **ECG-FM.** The checkpoint pretrained by self-supervision on MIMIC-IV-ECG and the PhysioNet/Computing in Cardiology Challenge 2021 collection, not a checkpoint fine-tuned on labels.
  The sizes of the network are taken from the checkpoint itself (12 transformer layers, 768 dimensions). The `.yaml` file next to it on Hugging Face describes a larger network, does not belong to this checkpoint and is not needed.
- The pretraining of HuBERT-ECG and ECG-FM had included some of the evaluation data without labels (PTB-XL, MIMIC-IV-ECG and, for HuBERT-ECG, CODE); EchoNext had not been seen by any model.

The cell below downloads what is missing (about 1.8 GB) with `git` and `huggingface_hub`, which is installed together with `transformers`.
Without them, download the files from the pages above by hand (on Hugging Face: *Files and versions*, or
`https://huggingface.co/<repository>/resolve/main/<file>`) and place them as in the table.

In [ ]:
import subprocess


def _download(repository, name, folder):
    """Download one file of a Hugging Face repository into `folder`, unless it is there already."""
    if not (folder / name).exists():
        from huggingface_hub import hf_hub_download
        hf_hub_download(repository, name, local_dir=folder)


if not (ECGFOUNDER_CODE / "net1d.py").exists():
    subprocess.run(["git", "clone", "https://github.com/PKUDigitalHealth/ECGFounder", str(ECGFOUNDER_CODE)], check=True)
_download("PKUDigitalHealth/ECGFounder", ECGFOUNDER_WEIGHTS.name, ECGFOUNDER_WEIGHTS.parent)
_download("Edoardo-Coppola/hubert-ecg-base", "model.safetensors", HUBERT_DIR)
_download("Edoardo-Coppola/hubert-ecg-base", "config.json", HUBERT_DIR)
_download("wanglab/ecg-fm", ECGFM_WEIGHTS.name, ECGFM_WEIGHTS.parent)

### Check of the files

The SHA-256 values are those of the files that the study used. A different value means that the file on the server has changed since.

In [ ]:
import hashlib

_EXPECTED = {ECGFOUNDER_WEIGHTS: "ee199f3781f4ae1f732973267f003da0a759ea12bddb0dd28a77faa60aca7997",
             HUBERT_DIR / "model.safetensors": "05bc1b1317f8e3063811a03fb840f5bf8a85968191e209c0cfbaa0c52c8aa1ae",
             ECGFM_WEIGHTS: "4d0142bcb485eb9f0c7845e0c19ff3463f6ae9d0e458eab69136efe90ceb9b7e"}
for _file, _sha in _EXPECTED.items():
    _h = hashlib.sha256()
    with open(_file, "rb") as _f:
        for _block in iter(lambda: _f.read(1 << 24), b""):
            _h.update(_block)
    print(f"{_file.relative_to(PRETRAINED).as_posix():45s} {_file.stat().st_size / 1e9:.2f} GB  SHA-256 {'as in the study' if _h.hexdigest() == _sha else 'DIFFERS from the file of the study'}")

### The three models with a five-output head

`FoundationModel(name)` (notebook 00) builds a network, loads the released weights and adds a linear layer with five outputs on the embedding.
The cell below builds each model once and reports how many tensors of the released file were loaded, whether any tensor of the network was left without pretrained values (`missing`),
and the number of parameters. Tensors that only pretraining uses (prediction heads, quantizer, mask token) are not loaded.
In HuBERT-ECG and ECG-FM the convolutional feature extractor is fixed, so fewer parameters are trainable than exist.
The ECGFounder network keeps its original output layer for 150 labels (0.15 million parameters), which is not used here.

In [ ]:
_rows = []
for _name in MODELS:
    _net = FoundationModel(_name); _r = _net.load_report
    _rows.append({"model": MODEL_NAMES[_name], "tensors loaded": _r["loaded"], "missing": len(_r["missing"]), "not used": len(_r["not_used"]), "embedding": _net.dim,
                  "parameters (million)": sum(p.numel() for p in _net.parameters()) / 1e6, "trainable (million)": sum(p.numel() for p in _net.parameters() if p.requires_grad) / 1e6})
    assert not _r["missing"], _r["missing"]
    del _net
pd.DataFrame(_rows).round(2)

## How the models are fine-tuned

**Task and examples.** The five outputs are the diagnostic superclasses of PTB-XL (`CLASSES`); a recording can have several. Training uses folds 1–8 and validation fold 9 (notebook 02).
A model reads one 5-s segment as its ECG, so the two adjacent segments of each training recording are separate examples with the labels of the recording.

**Common frame.** At every step 128 training recordings are drawn with replacement, and one of the two segments of each is taken at random.
The loss is the binary cross-entropy on the five outputs, with the positive class of each diagnosis weighted by (1 − prevalence) / prevalence.
The optimizer is AdamW with weight decay 0.01 and a one-cycle schedule of the learning rate with 10% warm-up, for 3,000 steps, with mixed precision.
Every 250 steps and after the last step, the macro AUROC of the five classes in the validation fold is calculated for the first and for the second segment and averaged,
and the weights of the step with the highest value are kept. Every configuration is trained with three seeds.

**Settings that differ between the foundation models**

| | ECGFounder | HuBERT-ECG | ECG-FM |
|---|---|---|---|
| Trained layers | all | transformer layers; convolutional feature extractor fixed | transformer layers; convolutional feature extractor fixed |
| Learning rate of the network | 1 × 10⁻⁴ | 1 × 10⁻⁵ | 1 × 10⁻⁵ |
| Learning rate of the head | 1 × 10⁻³ | 1 × 10⁻³ | 1 × 10⁻³ |
| Dropout during fine-tuning | none | none | 0.1 after the activation of the feed-forward layers |
| Input (details in notebook 00) | 500 Hz, standardized over all leads and time | 100 Hz, each lead scaled to −1 … 1 | 500 Hz, each lead standardized |

The trained layers follow the published fine-tuning of each model. For HuBERT-ECG and ECG-FM, candidate learning rates of the network were compared in one baseline run each,
and the rate with the highest validation AUROC was used. The transformers are trained with gradient checkpointing, which gives the same gradients with less memory, so that distillation fits into 10 GB.

**Kinds of fine-tuned model**

| Kind | Input in training | Added to the loss | Role |
|---|---|---|---|
| `baseline` | whole segment | – | the model as it is usually adapted |
| `teacher` | a random 2.5-s window of the segment | – | source of the distillation target; not used for reading |
| `student` | whole segment | distillation from the teacher of the same seed | the training remedy |
| `windows` | a random 2.5-s window of the segment | – | alternative: training on windows |
| `agreement` | whole segment | agreement loss between two shifted views | alternative: direct agreement loss |
| `selfdistill` | whole segment | distillation from the baseline model of the same seed | alternative: self-distillation |
| `dropoutNN` | whole segment | – | baseline trained with dropout at rate 0.NN, for Monte Carlo dropout |

*Teacher.* The teacher is the same foundation model fine-tuned on random 2.5-s windows: at every step a window with a random onset is cut from each drawn segment.
In ECGFounder only the last two of the seven stages of the teacher, and its head, are trained.
In HuBERT-ECG and ECG-FM the teacher has the same trained layers as the baseline. For the selection of the checkpoint, a model trained on windows is validated on four fixed 2.5-s windows of each validation recording.

*Distillation.* For each segment, the target is the mean logit of the teacher over seven evenly spaced 2.5-s windows of the segment (onsets from 0 to 2.5 s).
The student reads the whole segment once. Each step adds 128 unlabelled segments from the MIMIC-IV-ECG pool, and the loss is

> cross-entropy(labels) + ½ × [ MSE(student logits, target) in the labelled segments + MSE(student logits, target) in the unlabelled segments ]

so the student learns to give in one read of the segment what the teacher gives on average over the windows. The targets do not change during training and are calculated once, before the student is trained.
After training, the teacher is no longer needed.

*Agreement loss.* Each step adds 64 training segments and 64 pool segments, each as two 4.5-s views whose onsets are drawn independently between 0 and 0.5 s of the segment.
For each class the loss is the variance of the logit within the pairs of views, divided by the sum of the variances within and between the pairs; it is averaged over the classes and over the two sources and added with weight 1.

*Dropout.* ECGFounder is fine-tuned without dropout, as in its repository. For Monte Carlo dropout, baseline models are trained with the dropout layers of the residual blocks (after each activation) switched on.

## Jobs

A job is (model, kind, seed). The order matters only within a seed: a `student` needs the `teacher` and a `selfdistill` model needs the `baseline` of the same seed.

In [ ]:
MAIN_JOBS = [(model, kind, seed) for model in MODELS for seed in SEEDS for kind in ("baseline", "teacher", "student")]
ALTERNATIVE_JOBS = [("ecgfounder", kind, seed) for kind in ("windows", "agreement", "selfdistill") for seed in SEEDS]
ENSEMBLE_JOBS = [("ecgfounder", "baseline", seed) for seed in ENSEMBLE_SEEDS if seed not in SEEDS]
DROPOUT_JOBS = [("ecgfounder", f"dropout{round(100 * rate)}", seed) for rate in DROPOUT_RATES for seed in SEEDS]

if QUICK_TEST:            # one job of every kind
    MAIN_JOBS = [j for j in MAIN_JOBS if j[2] == 0]; ALTERNATIVE_JOBS = [j for j in ALTERNATIVE_JOBS if j[2] == 0]; ENSEMBLE_JOBS = []; DROPOUT_JOBS = DROPOUT_JOBS[:1]
print(len(MAIN_JOBS) + len(ALTERNATIVE_JOBS) + len(ENSEMBLE_JOBS) + len(DROPOUT_JOBS), "jobs:", len(MAIN_JOBS), "main,", len(ALTERNATIVE_JOBS), "alternatives,",
      len(ENSEMBLE_JOBS), "ensemble,", len(DROPOUT_JOBS), "dropout")


def run_jobs(jobs):
    """Train the models of the list that do not exist yet. Before a model that is trained by distillation, the targets are calculated if they do not exist yet."""
    for model, kind, seed in jobs:
        tag = model_tag(model, kind, seed)
        if (MODEL_DIR / f"{tag}.pt").exists():
            print(f"[{tag}] exists: skipped"); continue
        source = kind_settings(model, kind)["target_from"]            # "teacher" for a student, "baseline" for self-distillation, otherwise None
        if source is not None:
            source_tag = model_tag(model, source, seed)
            if not (TARGET_DIR / f"{source_tag}.npz").exists():
                compute_teacher_targets(source_tag, subset=SUBSET, model_dir=MODEL_DIR, target_dir=TARGET_DIR)
        finetune(model, kind, seed, steps=STEPS, eval_every=EVERY, subset=SUBSET, model_dir=MODEL_DIR, log_dir=LOG_DIR, target_dir=TARGET_DIR)

## Baseline, teacher, teacher targets and student

For each foundation model and seed: the baseline, the teacher, the targets of the teacher for the 34,168 training segments and the 42,722 segments of the unlabelled pool
(`compute_teacher_targets`), and the student. About 20 min for a seed of ECGFounder, 65 min for HuBERT-ECG and 90 min for ECG-FM.

In [ ]:
run_jobs(MAIN_JOBS)                # about 9 h on one RTX 3080

## Comparison models in ECGFounder

### Alternatives for the training remedy

Training on 2.5-s windows, the agreement loss, and self-distillation, all with every layer trained (about 7, 15 and 19 min for a model; self-distillation first calculates the targets of the baseline model, 2 min).

In [ ]:
run_jobs(ALTERNATIVE_JOBS)         # about 2 h

### Further baseline models for the ensemble

Seven baseline models with seeds 3 to 9, trained exactly like those with seeds 0 to 2 (about 6 to 9 min each). The ensemble averages the probabilities of the ten models (notebook 05).

In [ ]:
run_jobs(ENSEMBLE_JOBS)            # about 1 h

### Baseline models trained with dropout

Dropout rates 0.1 to 0.5, three seeds each (about 5 to 10 min each). The rate with the highest validation AUROC is used (notebook 05).

In [ ]:
run_jobs(DROPOUT_JOBS)             # about 2 h

## Training logs

The table lists, for every log in `TRAINING_LOGS`, the selected step, its validation AUROC and the time of the run.
Steps are counted from 0, so the last step of a run of 3,000 steps is 2,999.

In [ ]:
_rows = []
for _f in sorted(LOG_DIR.glob("*.csv")):
    _log = pd.read_csv(_f); _best = _log.loc[_log.val_auroc.idxmax()]
    _rows.append({"model": _f.stem, "selected step": int(_best.step), "validation AUROC": _best.val_auroc, "last step": int(_log.step.iloc[-1]), "minutes": _log.minutes.iloc[-1]})
LOGS = pd.DataFrame(_rows)
LOGS.round(4)